In [2]:
import os
os.listdir()



['.ipynb_checkpoints',
 '2015.csv',
 '2016.csv',
 '2017.csv',
 '2018.csv',
 '2019.csv',
 'PM 07 Ex 4.ipynb',
 'Reviews.csv']

In [3]:
import os, json, re
from datetime import datetime
import pandas as pd
import numpy as np

# NLP imports
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import RegexpTokenizer

# sklearn for optional text features
from sklearn.feature_extraction.text import CountVectorizer

##### Ensure required NLTK resources (avoid punkt_tab)

In [4]:
# We will use RegexpTokenizer to avoid punkt_tab issues; still need stopwords + vader_lexicon
for pkg in ["stopwords", "vader_lexicon"]:
    try:
        nltk.data.find(f"corpora/{pkg}")
    except LookupError:
        nltk.download(pkg)


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Geeks_PC31\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [5]:
# 1. LOAD STRUCTURED CSV FILES (2015-2019)

files_to_load = ["2015.csv", "2016.csv", "2017.csv", "2018.csv", "2019.csv"]
loaded = []
for f in files_to_load:
    if os.path.exists(f):
        try:
            df_year = pd.read_csv(f)
            df_year["source_file"] = f
            loaded.append(df_year)
            print(f"Loaded {f} shape={df_year.shape}")
        except Exception as e:
            print(f"Error loading {f}: {e}")
    else:
        print(f"File not found (skipped): {f}")

if len(loaded) == 0:
    raise FileNotFoundError("No structured year CSVs loaded. Make sure 2015.csv ... 2019.csv exist in the working directory.")

df_struct = pd.concat(loaded, ignore_index=True)
print("\nCombined structured shape:", df_struct.shape)

Loaded 2015.csv shape=(158, 13)
Loaded 2016.csv shape=(157, 14)
Loaded 2017.csv shape=(155, 13)
Loaded 2018.csv shape=(156, 10)
Loaded 2019.csv shape=(156, 10)

Combined structured shape: (782, 31)


### 2. STRUCTURED DATA CLEANING

In [6]:
df_s = df_struct.copy()

# Standardize column names: strip, lowercase, underscores
df_s.columns = [c.strip().lower().replace(" ", "_") for c in df_s.columns]
print("\nStructured columns:", df_s.columns.tolist())

# Trim whitespace for object columns
for c in df_s.select_dtypes(include="object").columns:
    df_s[c] = df_s[c].astype(str).str.strip()


Structured columns: ['country', 'region', 'happiness_rank', 'happiness_score', 'standard_error', 'economy_(gdp_per_capita)', 'family', 'health_(life_expectancy)', 'freedom', 'trust_(government_corruption)', 'generosity', 'dystopia_residual', 'source_file', 'lower_confidence_interval', 'upper_confidence_interval', 'happiness.rank', 'happiness.score', 'whisker.high', 'whisker.low', 'economy..gdp.per.capita.', 'health..life.expectancy.', 'trust..government.corruption.', 'dystopia.residual', 'overall_rank', 'country_or_region', 'score', 'gdp_per_capita', 'social_support', 'healthy_life_expectancy', 'freedom_to_make_life_choices', 'perceptions_of_corruption']


In [7]:
# Detect numeric columns and impute small missingness with median
num_cols = df_s.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    missing_pct = df_s[col].isna().mean()
    if missing_pct > 0 and missing_pct <= 0.05:
        med = df_s[col].median()
        df_s[col] = df_s[col].fillna(med)
        print(f"Imputed median for {col} (pct missing {missing_pct:.3f})")
    elif missing_pct > 0.05:
        df_s[col + "_missing_flag"] = df_s[col].isna().astype(int)
        df_s[col] = df_s[col].fillna(df_s[col].median())
        print(f"Added missing flag and median-imputed for {col} (pct missing {missing_pct:.3f})")

Added missing flag and median-imputed for happiness_rank (pct missing 0.597)
Added missing flag and median-imputed for happiness_score (pct missing 0.597)
Added missing flag and median-imputed for standard_error (pct missing 0.798)
Added missing flag and median-imputed for economy_(gdp_per_capita) (pct missing 0.597)
Added missing flag and median-imputed for family (pct missing 0.399)
Added missing flag and median-imputed for health_(life_expectancy) (pct missing 0.597)
Added missing flag and median-imputed for freedom (pct missing 0.399)
Added missing flag and median-imputed for trust_(government_corruption) (pct missing 0.597)
Added missing flag and median-imputed for dystopia_residual (pct missing 0.597)
Added missing flag and median-imputed for lower_confidence_interval (pct missing 0.799)
Added missing flag and median-imputed for upper_confidence_interval (pct missing 0.799)
Added missing flag and median-imputed for happiness.rank (pct missing 0.802)
Added missing flag and median-

In [8]:
# Detect and parse a date column (first column name containing 'date')
date_col = None
for c in df_s.columns:
    if "date" in c:
        date_col = c
        try:
            df_s[date_col] = pd.to_datetime(df_s[date_col], errors="coerce")
            print(f"Parsed date column: {date_col}")
        except Exception:
            print(f"Found {date_col} but failed to parse as datetime.")
        break

if date_col:
    df_s["year_parsed"] = df_s[date_col].dt.year
    df_s["month_parsed"] = df_s[date_col].dt.month

In [9]:
# Remove duplicate rows
before = len(df_s)
df_s = df_s.drop_duplicates()
after = len(df_s)
print(f"Removed duplicates: {before-after}")

Removed duplicates: 0


In [10]:
# Save cleaned structured file
happiness_out = "happiness_clean.csv"
df_s.to_csv(happiness_out, index=False)
print(f"Saved cleaned structured data -> {happiness_out}")

Saved cleaned structured data -> happiness_clean.csv


### 3. LOAD & CLEAN UNSTRUCTURED REVIEWS

In [12]:
reviews_file = "Reviews.csv"
if not os.path.exists(reviews_file):
    raise FileNotFoundError("Reviews.csv not found in working directory.")

df_reviews = pd.read_csv(reviews_file, encoding="utf-8", low_memory=False)
print("\nLoaded reviews shape:", df_reviews.shape)
print("Review columns:", df_reviews.columns.tolist())



Loaded reviews shape: (568454, 10)
Review columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


In [13]:
# Detect the most likely text column
candidate_text_cols = ["reviewtext", "text", "review", "review_body", "comments", "summary"]
text_col = None
for c in df_reviews.columns:
    if c.lower() in candidate_text_cols or any(k in c.lower() for k in candidate_text_cols):
        text_col = c
        break
if text_col is None:
    # fallback: choose the longest average object column
    obj_cols = [c for c in df_reviews.columns if df_reviews[c].dtype == object]
    if len(obj_cols) == 0:
        raise ValueError("No text-like column found in Reviews.csv")
    avg_len = {c: df_reviews[c].dropna().astype(str).map(len).mean() for c in obj_cols}
    text_col = max(avg_len, key=avg_len.get)

print("Detected review text column:", text_col)

Detected review text column: Summary


In [14]:
# Text cleaning utilities (safe tokenizer that doesn't require punkt)
tokenizer = RegexpTokenizer(r"\w+")
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
sia = SentimentIntensityAnalyzer()

def clean_text_pipeline(raw):
    if pd.isna(raw):
        raw = ""
    s = str(raw)
    s = s.lower()
    s = re.sub(r"http\S+|www\S+|https\S+", " ", s)            # remove URLs
    s = re.sub(r"[^a-z0-9\s']", " ", s)                      # keep alphanumerics and apostrophes
    s = re.sub(r"\s+", " ", s).strip()
    tokens = tokenizer.tokenize(s)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    stems = [stemmer.stem(t) for t in tokens]
    cleaned_text = " ".join(stems)
    sentiment = sia.polarity_scores(" ".join(tokens))
    return cleaned_text, sentiment

In [15]:
# Apply cleaning (vectorized-ish)
cleaned_texts = []
sentiments = []
for i, raw in enumerate(df_reviews[text_col].fillna("")):
    cleaned, sent = clean_text_pipeline(raw)
    cleaned_texts.append(cleaned)
    sentiments.append(sent)
    if (i+1) % 5000 == 0:
        print(f"Processed {i+1} reviews...")

df_reviews["_cleaned_text"] = cleaned_texts
df_reviews["_sentiment"] = sentiments
df_reviews["_sentiment_compound"] = df_reviews["_sentiment"].apply(lambda d: d.get("compound", np.nan))
df_reviews["_sentiment_pos"] = df_reviews["_sentiment"].apply(lambda d: d.get("pos", np.nan))
df_reviews["_sentiment_neu"] = df_reviews["_sentiment"].apply(lambda d: d.get("neu", np.nan))
df_reviews["_sentiment_neg"] = df_reviews["_sentiment"].apply(lambda d: d.get("neg", np.nan))


Processed 5000 reviews...
Processed 10000 reviews...
Processed 15000 reviews...
Processed 20000 reviews...
Processed 25000 reviews...
Processed 30000 reviews...
Processed 35000 reviews...
Processed 40000 reviews...
Processed 45000 reviews...
Processed 50000 reviews...
Processed 55000 reviews...
Processed 60000 reviews...
Processed 65000 reviews...
Processed 70000 reviews...
Processed 75000 reviews...
Processed 80000 reviews...
Processed 85000 reviews...
Processed 90000 reviews...
Processed 95000 reviews...
Processed 100000 reviews...
Processed 105000 reviews...
Processed 110000 reviews...
Processed 115000 reviews...
Processed 120000 reviews...
Processed 125000 reviews...
Processed 130000 reviews...
Processed 135000 reviews...
Processed 140000 reviews...
Processed 145000 reviews...
Processed 150000 reviews...
Processed 155000 reviews...
Processed 160000 reviews...
Processed 165000 reviews...
Processed 170000 reviews...
Processed 175000 reviews...
Processed 180000 reviews...
Processed 18

In [16]:
# Save cleaned reviews to newline-delimited JSON
reviews_out = "reviews_clean.json"
df_reviews.to_json(reviews_out, orient="records", lines=True, force_ascii=False)
print(f"Saved cleaned reviews -> {reviews_out}")

Saved cleaned reviews -> reviews_clean.json


In [17]:
# 4. SCRIPTING DEMO (variables, operators, conditionals, loops, functions)

print("\n=== Scripting demo outputs ===")
a = 5
b = 7
def add(x,y): return x+y
print("add(a,b) =", add(a,b))
if a < b:
    rel = "a is less than b"
else:
    rel = "a is not less than b"
print("Conditional result:", rel)



=== Scripting demo outputs ===
add(a,b) = 12
Conditional result: a is less than b


In [18]:
# loop example: sum first 10 positive integers
s = 0
for i in range(1,11):
    s += i
print("Sum 1..10 =", s)

Sum 1..10 = 55


In [19]:
# 5. PROBABILITY & BASIC INFERENCE EXAMPLE
print("\n=== Probability example ===")
# use 'is_holiday' or 'isholiday' column if exists (case-insensitive)
ih_col = None
for c in df_s.columns:
    if c.lower() == "isholiday" or c.lower() == "is_holiday":
        ih_col = c
        break

sales_col_candidates = [c for c in df_s.columns if "sales" in c.lower() or "weekly_sales" in c.lower()]
sales_col = sales_col_candidates[0] if sales_col_candidates else None

if ih_col and sales_col:
    median_sales = df_s[sales_col].median()
    p_h = (df_s[df_s[ih_col] == True][sales_col] > median_sales).mean()
    p_n = (df_s[df_s[ih_col] == False][sales_col] > median_sales).mean()
    print(f"P(sales > median | holiday) = {p_h:.3f}")
    print(f"P(sales > median | non-holiday) = {p_n:.3f}")
    print("Difference:", round(p_h - p_n, 3))
else:
    print("Cannot compute holiday probability example: missing IsHoliday or sales column in structured data.")



=== Probability example ===
Cannot compute holiday probability example: missing IsHoliday or sales column in structured data.


In [20]:
# 6. QUICK EXPLORATORY OUTPUT (samples)

print("\nSample structured data (first 3 rows):")
display(df_s.head(3))

print("\nSample cleaned reviews (first 3 rows):")
display(df_reviews[[text_col, "_cleaned_text", "_sentiment_compound"]].head(3))


Sample structured data (first 3 rows):


,country,region,happiness_rank,happiness_score,standard_error,economy_(gdp_per_capita),family,health_(life_expectancy),freedom,trust_(government_corruption),...,health..life.expectancy._missing_flag,trust..government.corruption._missing_flag,dystopia.residual_missing_flag,overall_rank_missing_flag,score_missing_flag,gdp_per_capita_missing_flag,social_support_missing_flag,healthy_life_expectancy_missing_flag,freedom_to_make_life_choices_missing_flag,perceptions_of_corruption_missing_flag
0,Switzerland,Western Europe,1.0,7.587,0.03411,1.39651,1.34951,0.94143,0.66557,0.41978,...,1,1,1,1,1,1,1,1,1,1
1,Iceland,Western Europe,2.0,7.561,0.04884,1.30232,1.40223,0.94784,0.62877,0.14145,...,1,1,1,1,1,1,1,1,1,1
2,Denmark,Western Europe,3.0,7.527,0.03328,1.32548,1.36058,0.87464,0.64938,0.48357,...,1,1,1,1,1,1,1,1,1,1



Sample cleaned reviews (first 3 rows):


,Summary,_cleaned_text,_sentiment_compound
0,Good Quality Dog Food,good qualiti dog food,0.4404
1,Not as Advertised,advertis,0.0000
2,"""Delight"" says it all",delight say,0.5994


In [21]:
# 7. PRE-PROCESSING REPORT (printed)

print("\n\n=== PRE-PROCESSING REPORT ===")
print("Structured cleaning steps applied:")
print("- Combined 2015-2019 CSVs")
print("- Standardized column names (lowercase + underscores)")
print("- Trimmed whitespace in text fields")
print("- Numeric missing values median-imputed; missing flags added where >5%")
print("- Parsed date column (if present) and created year/month")
print("- Removed duplicates")
print(f"- Output saved: {happiness_out}")

print("\nUnstructured cleaning steps applied:")
print("- Detected review text column automatically")
print("- Lowercased text, removed URLs and non-alphanumerics")
print("- Tokenized using RegexpTokenizer (no punkt dependency)")
print("- Removed stopwords, applied Porter stemming")
print("- Computed VADER sentiment scores (compound, pos, neu, neg)")
print(f"- Output saved: {reviews_out}")

print("\nAll done — files written to the working directory.")



=== PRE-PROCESSING REPORT ===
Structured cleaning steps applied:
- Combined 2015-2019 CSVs
- Standardized column names (lowercase + underscores)
- Trimmed whitespace in text fields
- Numeric missing values median-imputed; missing flags added where >5%
- Parsed date column (if present) and created year/month
- Removed duplicates
- Output saved: happiness_clean.csv

Unstructured cleaning steps applied:
- Detected review text column automatically
- Lowercased text, removed URLs and non-alphanumerics
- Tokenized using RegexpTokenizer (no punkt dependency)
- Removed stopwords, applied Porter stemming
- Computed VADER sentiment scores (compound, pos, neu, neg)
- Output saved: reviews_clean.json

All done — files written to the working directory.
